# W&B Debug Notebook

Use this notebook to verify Weights & Biases setup in this workspace.

In [ ]:
import math
import os
import random
import time
from datetime import datetime
from pathlib import Path

In [ ]:
# Debug configuration
repo_root = Path.cwd()
project = "taiko-transformer"
entity = "yiy523-lehigh-university"
modelname = "mock-model"
runname = "wandb-debug"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

enable_wandb = True
# Set True if you want to test without internet/auth.
wandb_offline = False
wandb_api_key = "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"
wandb_notebook_name = "wandb_debug.ipynb"

os.environ["WANDB_NOTEBOOK_NAME"] = wandb_notebook_name
os.environ.setdefault("WANDB_DIR", str((repo_root / ".wandb").resolve()))
Path(os.environ["WANDB_DIR"]).mkdir(parents=True, exist_ok=True)

if enable_wandb:
    import wandb
    if wandb_offline:
        os.environ["WANDB_MODE"] = "offline"
    else:
        # Prevent stale offline state from previous notebook runs.
        os.environ.pop("WANDB_MODE", None)
        if wandb_api_key.strip():
            os.environ["WANDB_API_KEY"] = wandb_api_key.strip()
        api_key = os.environ.get("WANDB_API_KEY", "").strip()
        if not api_key:
            raise RuntimeError(
                "WANDB_API_KEY is not set in notebook config or environment. "
                "Set wandb_api_key above, or set wandb_offline=True."
            )
        try:
            wandb.login(key=api_key, relogin=True)
        except Exception as exc:
            raise RuntimeError(
                f"W&B login failed: {exc}. If you only want to test logging locally, set wandb_offline=True."
            ) from exc
    print(f"wandb.__version__={wandb.__version__}")
else:
    print("W&B disabled: skipping wandb import/login.")

print(f"WANDB_NOTEBOOK_NAME={os.environ.get('WANDB_NOTEBOOK_NAME')}")
print(f"WANDB_DIR={os.environ.get('WANDB_DIR')}")
print(f"WANDB_MODE={os.environ.get('WANDB_MODE', 'online')}")

In [ ]:
run = None
if enable_wandb:
    try:
        run = wandb.init(
            project=project,
            name=f" taiko-transformer_run_{modelname}_{runname}_{timestamp}",
            entity=entity,
            settings=wandb.Settings(init_timeout=120),
        )
    except Exception as exc:
        raise RuntimeError(
            f"wandb.init failed: {exc}. Try wandb_offline=True to validate local logging path first."
        ) from exc

    run.define_metric("global_step")
    run.define_metric("*", step_metric="global_step")
    print("W&B run started.")
    print(f"run.id={run.id}")
    print(f"run.url={run.url}")
else:
    print("W&B disabled: no run initialized.")

In [ ]:
# Mock training loop
random.seed(42)
for step in range(1, 51):
    train_loss = 2.5 * math.exp(-step / 20.0) + random.random() * 0.05
    val_loss = 2.6 * math.exp(-step / 22.0) + random.random() * 0.07
    lr = 1e-3 * (0.98 ** step)

    if enable_wandb:
        wandb.log(
            {
                "global_step": step,
                "epoch": step / 10.0,
                "train/loss_batch": train_loss,
                "val/loss_batch": val_loss,
                "optimizer/lr": lr,
            }
        )
    time.sleep(0.02)

if enable_wandb:
    wandb.log(
        {
            "global_step": 50,
            "train/loss_epoch": train_loss,
            "val/loss_epoch": val_loss,
            "checkpoint/last_path": str((repo_root / "checkpoints" / "debug" / "last.ckpt").resolve()),
            "checkpoint/best_updated": 1,
        }
    )

print("Mock logs sent.")

In [ ]:
if enable_wandb and run is not None:
    run.finish()
    print("W&B run finished.")
else:
    print("W&B was disabled; nothing to finish.")